# Practical 10: Python Visual EDA
**Problem Statement:** Generate publication-ready statistical plots.

This notebook will automatically create the required directories, generate mock data (including Sales and Profit) to simulate the environment, and output 8 statistical charts to the `outputs/plots/` directory.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Attractive chart styling
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 220
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 13
palette = sns.color_palette('Set2')

# Create directory for outputs if it doesn't exist
output_dir = '../outputs/plots/'
os.makedirs(output_dir, exist_ok=True)

def save_chart(filename):
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename), bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()

# Generate Mock Data (to simulate Practical 9 state)
np.random.seed(42)
dates = pd.date_range(start='2023-01-01', periods=200)
df = pd.DataFrame({
    'Date': dates,
    'Sales': np.random.normal(1000, 250, 200),
    'Profit': np.random.normal(150, 80, 200),
    'Discount': np.random.uniform(0, 0.4, 200),
    'Category': np.random.choice(['Technology', 'Furniture', 'Supplies'], 200),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], 200)
})

# Inject mathematical outliers for the boxplot
df.loc[10, 'Profit'] = 800
df.loc[45, 'Profit'] = -400

# Inject missing values for the missing-value chart
df.loc[100:110, 'Sales'] = np.nan

# Feature engineering for richer visual analysis
df['Month'] = df['Date'].dt.to_period('M').astype(str)
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['Profit_Margin'] = (df['Profit'] / df['Sales']) * 100
df['Sales_Band'] = pd.cut(df['Sales'], bins=[0, 750, 1000, 1250, np.inf], labels=['Low', 'Medium', 'High', 'Very High'])

monthly_summary = df.groupby('Month', as_index=False).agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Avg_Discount=('Discount', 'mean')
)

print('Environment setup complete. Data loaded with extra visual EDA features.')
display(df.head())


In [ ]:
# Chart 1: Distribution Plot (Histogram) for 'Sales'
plt.figure(figsize=(8, 5))
sns.histplot(df['Sales'].dropna(), kde=True)
plt.title('Sales Distribution')
save_chart('01_histogram.png')

**Observation 1:** The 'Sales' data displays a roughly normal distribution centered around the 1000 mark, indicating stable average transaction sizes.

In [ ]:
# Chart 2: Boxplot for 'Profit' to identify outliers
plt.figure(figsize=(8, 3))
sns.boxplot(x=df['Profit'])
plt.title('Profit Boxplot (Outlier Detection)')
save_chart('02_boxplot.png')

**Observation 2:** The boxplot mathematically identifies several extreme outliers in 'Profit', both on the highly profitable end and heavily negative (loss) end.

In [ ]:
# Chart 3: Correlation Heatmap
# Recreating the correlation matrix from Practical 9
corr_matrix = df[['Sales', 'Profit', 'Discount']].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
save_chart('03_heatmap.png')

**Observation 3:** The heatmap reveals minimal linear correlation between Sales, Profit, and Discount in this dataset, indicating other factors drive profitability.

In [ ]:
# Chart 4: Bar Chart
plt.figure(figsize=(8, 5))
sns.barplot(x='Category', y='Sales', data=df, errorbar=None)
plt.title('Average Sales by Category')
save_chart('04_barchart.png')

**Observation 4:** Average sales are relatively evenly distributed across Technology, Furniture, and Supplies categories.

In [ ]:
# Chart 5: Line / Trend Chart
plt.figure(figsize=(10, 5))
sns.lineplot(x='Date', y='Sales', data=df.dropna())
plt.title('Sales Trend Over Time')
plt.xticks(rotation=45)
save_chart('05_linechart.png')

**Observation 5:** Sales volume exhibits high daily volatility over time without a strictly defined upward or downward long-term trend.

In [ ]:
# Chart 6: Scatter Plot
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Sales', y='Profit', hue='Category', data=df)
plt.title('Sales vs. Profit by Category')
save_chart('06_scatterplot.png')

**Observation 6:** The scatter plot shows a cluster around the mean with no distinct linear trajectory connecting higher sales volume directly to higher profit.

In [ ]:
# Chart 7: Missing-Value Chart
plt.figure(figsize=(8, 5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Data Map')
save_chart('07_missingvalues.png')

**Observation 7:** A continuous block of missing data is visually apparent in the 'Sales' column, indicating a potential temporary data collection failure.

In [ ]:
# Chart 8: Grouped Comparison Chart
plt.figure(figsize=(10, 5))
sns.boxplot(x='Category', y='Sales', hue='Region', data=df)
plt.title('Sales Distribution by Category and Region')
save_chart('08_groupedchart.png')

**Observation 8:** Grouping by region within categories shows that the spread and median sales are largely consistent across different geographic regions.

## Extra Visual EDA Features
The charts below add more attractive business-style views: pie/donut charts, category-region comparisons, trend summaries, and distribution diagnostics.


In [ ]:
# Chart 9: Pie Chart - Sales Share by Category
category_sales = df.groupby('Category')['Sales'].sum().dropna().sort_values(ascending=False)
plt.figure(figsize=(7, 7))
plt.pie(category_sales, labels=category_sales.index, autopct='%1.1f%%', startangle=90, colors=palette, wedgeprops={'edgecolor': 'white'})
plt.title('Sales Share by Category')
save_chart('09_pie_category_sales.png')


**Observation 9:** The pie chart quickly shows which product category contributes the largest share of total sales.


In [ ]:
# Chart 10: Donut Chart - Region Contribution
region_sales = df.groupby('Region')['Sales'].sum().dropna().sort_values(ascending=False)
plt.figure(figsize=(7, 7))
wedges, texts, autotexts = plt.pie(region_sales, labels=region_sales.index, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('Pastel1'), wedgeprops={'width': 0.42, 'edgecolor': 'white'})
plt.text(0, 0, 'Region\nSales', ha='center', va='center', fontsize=13, fontweight='bold')
plt.title('Regional Sales Contribution')
save_chart('10_donut_region_sales.png')


**Observation 10:** The donut chart makes regional contribution easy to compare while keeping the design cleaner than a standard pie chart.


In [ ]:
# Chart 11: Violin Plot - Profit Spread by Category
plt.figure(figsize=(9, 5))
sns.violinplot(x='Category', y='Profit', hue='Category', data=df, palette='Set2', inner='quartile', legend=False)
plt.axhline(0, color='black', linewidth=1, alpha=0.6)
plt.title('Profit Distribution by Category')
save_chart('11_violin_profit_category.png')


**Observation 11:** The violin plot shows both spread and density, making profit variability clearer than a simple average.


In [ ]:
# Chart 12: Monthly Sales and Profit Trend
plt.figure(figsize=(11, 5))
sns.lineplot(x='Month', y='Sales', data=monthly_summary, marker='o', label='Sales')
sns.lineplot(x='Month', y='Profit', data=monthly_summary, marker='o', label='Profit')
plt.title('Monthly Sales and Profit Trend')
plt.xticks(rotation=45)
plt.ylabel('Amount')
plt.legend()
save_chart('12_monthly_sales_profit.png')


**Observation 12:** Monthly aggregation smooths daily noise and makes broader business movement easier to interpret.


In [ ]:
# Chart 13: Stacked Bar Chart - Category Sales by Region
category_region = pd.pivot_table(df, values='Sales', index='Category', columns='Region', aggfunc='sum')
ax = category_region.plot(kind='bar', stacked=True, figsize=(10, 6), color=sns.color_palette('Set3', n_colors=4))
plt.title('Stacked Sales by Category and Region')
plt.xlabel('Category')
plt.ylabel('Total Sales')
plt.xticks(rotation=0)
plt.legend(title='Region', bbox_to_anchor=(1.02, 1), loc='upper left')
save_chart('13_stacked_category_region.png')


**Observation 13:** The stacked view compares total category performance while also showing how much each region contributes.


In [ ]:
# Chart 14: Pair Plot - Numeric Relationship Matrix
pair_df = df[['Sales', 'Profit', 'Discount', 'Category']].dropna().sample(n=min(120, df.dropna().shape[0]), random_state=42)
pair_grid = sns.pairplot(pair_df, hue='Category', diag_kind='hist', corner=True, palette='Set2', plot_kws={'s': 25, 'alpha': 0.75})
pair_grid.fig.suptitle('Pair Plot of Sales, Profit, and Discount', y=1.02, fontweight='bold')
pair_grid.savefig(os.path.join(output_dir, '14_pairplot_numeric_relationships.png'), bbox_inches='tight', dpi=180)
plt.show()
plt.close(pair_grid.fig)


**Observation 14:** The pair plot gives a compact matrix view of numeric relationships and category-wise clustering.


In [ ]:
# Chart 15: Profit Margin Distribution
plt.figure(figsize=(9, 5))
sns.histplot(df['Profit_Margin'].dropna(), bins=25, kde=True, color='#4C72B0')
plt.axvline(df['Profit_Margin'].median(), color='#C44E52', linestyle='--', label='Median margin')
plt.title('Profit Margin Distribution')
plt.xlabel('Profit Margin (%)')
plt.legend()
save_chart('15_profit_margin_distribution.png')


**Observation 15:** Profit margin distribution highlights how often transactions produce strong, weak, or negative profitability.


In [ ]:
# Chart 16: Sales Band Count Plot
plt.figure(figsize=(8, 5))
sns.countplot(x='Sales_Band', hue='Sales_Band', data=df, order=['Low', 'Medium', 'High', 'Very High'], palette='Set2', legend=False)
plt.title('Transaction Count by Sales Band')
plt.xlabel('Sales Band')
plt.ylabel('Number of Records')
save_chart('16_sales_band_count.png')


**Observation 16:** Sales bands convert raw values into business-friendly groups for quick transaction profiling.
